## PyINE code variables analysis

This notebook performs an analysis of code snippets in the TACO dataset to determine the frequency and overlap of variable defintions they contain.

In [ ]:
import collections
import pathlib
import pickle
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import tqdm
import wordcloud

import pyine.data.taco.dataset_utils as taco_utils
import pyine.data.traces.dataset_utils as traces_utils
import pyine.utils.code.blocks
import pyine.utils.code.variables
import pyine.utils.filesystem

## Analysis

In [ ]:
DATSET_NAME = "TACO"
DATASET_PATH = taco_utils.get_latest_repackaged_dataset_path()

# you can find the files referred to below here:
#    https://drive.google.com/drive/folders/12Wgw-AJ2xBwKmQEWDbmdAt7J0z7-XLKA
# copy them in the stats folder below to avoid recomputing everything (it takes hours)
STATS_DIR_PATH = pyine.utils.filesystem.get_data_root_path() / "stats"
CODE_BLOCK_STATS_FILE_NAME = STATS_DIR_PATH / "code_block_stats.pickle"
VARIABLE_METHOD_STATS_FILE_NAME = STATS_DIR_PATH / "variable_and_method_stats.pickle"
VARIABLE_INDEX_DB_FILE_NAME = STATS_DIR_PATH / "variable_index.sqlite"

In [ ]:
def apply_function_to_code_snippets(func_to_apply):
    count = 0
    errors = 0
    result = {}
    problem_iterator = traces_utils.CodingProblemIterator(
        dataset_name=DATSET_NAME,
        root_data_path=DATASET_PATH,
    )
    for problem, solutions in tqdm.tqdm(problem_iterator):
        for solution in solutions:
            try:
                current_result = func_to_apply(solution.code)
                result[str(solution.solution_id)] = current_result
                count += 1
            except SyntaxError:
                errors += 1
    print(f"found {errors} code snippets with errors")
    return result

In [ ]:
if pathlib.Path(CODE_BLOCK_STATS_FILE_NAME).is_file():
    print(f"stats already there for {CODE_BLOCK_STATS_FILE_NAME} - not computing again")
else:
    code_block_stats = apply_function_to_code_snippets(pyine.utils.code.blocks.identify_code_blocks)
    with open(CODE_BLOCK_STATS_FILE_NAME, "wb") as f:
        pickle.dump(code_block_stats, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if pathlib.Path(VARIABLE_METHOD_STATS_FILE_NAME).is_file():
    print(f"stats already there for {VARIABLE_METHOD_STATS_FILE_NAME} - not computing again")
else:
    variable_method_stats = apply_function_to_code_snippets(pyine.utils.code.variables.analyze_defs)
    with open(VARIABLE_METHOD_STATS_FILE_NAME, "wb") as f:
        pickle.dump(variable_method_stats, f, protocol=pickle.HIGHEST_PROTOCOL)

## Loading and displaying

In [ ]:
with open(CODE_BLOCK_STATS_FILE_NAME, "rb") as f:
    code_block_stats = pickle.load(f)
print(f"{len(code_block_stats)=:,}")

In [ ]:
with open(VARIABLE_METHOD_STATS_FILE_NAME, "rb") as f:
    variable_method_stats = pickle.load(f)
print(f"{len(variable_method_stats)=:,}")

## Plotting info on the variables

In [ ]:
keywords = [kw for d in variable_method_stats.values() for kw in d.get("bound_names", set())]


def norm(s: str) -> str:
    return s.strip().lower()


counts = collections.Counter(norm(k) for k in keywords if k.strip())
wc = wordcloud.WordCloud(width=1200, height=600, background_color="white")
wc = wc.generate_from_frequencies(counts)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
counts = collections.Counter(kw for d in variable_method_stats.values() for kw in d.get("bound_names", set()))
freqs = np.fromiter(counts.values(), dtype=int)

plt.figure(figsize=(10, 4))
plt.hist(freqs, bins="auto", log=True)  # log=True => y-axis log scale
plt.xscale("log")  # x-axis log scale
plt.xlabel("Frequency (how many top-level items contain the keyword) (log scale)")
plt.ylabel("Number of keywords (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
N = 50  # show only the most frequent N
counts = collections.Counter(kw for d in variable_method_stats.values() for kw in d.get("bound_names", set()))
top_items = counts.most_common(N)
labels = [k for k, _ in top_items]
freqs = [v for _, v in top_items]
x = np.arange(len(freqs))

plt.figure(figsize=(12, 5))
plt.bar(x, freqs)
plt.xticks(x, labels, rotation=45, ha="right", rotation_mode="anchor")
plt.xlabel("Keyword")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## Indexing the variables

In [ ]:
def build_kw_index(data: dict, db_path: str):
    p = pathlib.Path(db_path)
    if p.exists():
        p.unlink()  # rebuild cleanly
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.executescript(
        """
        PRAGMA journal_mode=WAL;
        PRAGMA synchronous=NORMAL;
        CREATE TABLE inv (
            term     TEXT NOT NULL,
            top_key  TEXT NOT NULL,
            PRIMARY KEY (term, top_key)
        ) WITHOUT ROWID;
        CREATE INDEX inv_term_idx ON inv(term);
    """
    )
    rows = ((kw, top_key) for top_key, d in data.items() for kw in d.get("bound_names", set()))
    cur.executemany("INSERT OR IGNORE INTO inv(term, top_key) VALUES (?, ?)", rows)
    con.commit()
    con.close()

In [ ]:
if pathlib.Path(VARIABLE_INDEX_DB_FILE_NAME).is_file():
    print(f"db already there for {VARIABLE_INDEX_DB_FILE_NAME} - not computing again")
else:
    build_kw_index(variable_method_stats, db_path=VARIABLE_INDEX_DB_FILE_NAME)

In [ ]:
def lookup_all(terms: list[str], db_path: str) -> list[str]:
    # return top_keys that contain *all* terms
    qmarks = ",".join("?" for _ in terms)
    sql = f"""
        SELECT top_key
        FROM inv
        WHERE term IN ({qmarks})
        GROUP BY top_key
        HAVING COUNT(DISTINCT term) = ?
    """
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute(sql, (*terms, len(terms)))
    out = [r[0] for r in cur.fetchall()]
    con.close()
    return out

In [ ]:
res = lookup_all(["self", "dog"], VARIABLE_INDEX_DB_FILE_NAME)
print(f"found {len(res)} results")